In [17]:
import codecs
import shutil
import numpy as np
import pandas as pd 
import os
import re

### CTan Morphometry Results 2D
##### Author: Anna Valentine (annavalentine@mines.edu)
#### Date: 07/19/21

#### Purpose:  
Takes in Ctan/batman files from microCT and concatonates the 2D results from the snowpit into one dataframe. This is meant for one snowpit at a time. 

In [3]:
# Write down my standard path here
#path = '/Users/annav/1_WORK_CRREL/CTan Project 1/' 
#files = path+'Data/*.txt'  
cwd = os.getcwd()
UTF8_folder = cwd + '/D_UTF8/'   # set up a folder for UTF8 conversion


In [4]:
os.getcwd()

'/Users/f006fk7/Library/CloudStorage/GoogleDrive-anna.m.valentine.th@dartmouth.edu/My Drive/crrel/mCT Processing/SnowEx2023_txt/CODE'

In [5]:
#general takes a path and spits out all of the file names 
def list_files_local(path):
    """ Get file list form local folder. """
    from glob import glob
    return glob(path)


In [6]:
## searching for my ctan files off of the harddrive: 
spath = '/Volumes/SnowEx23/SnowEx_mCT_2023/Uncasted/FB_Bonaza_EA410/*'
FB_Bonanza = '/Volumes/My Passport/Snowex_2023/Uncasted/Fairbanks/Bonanza/*'
FB_Creamers = '/Volumes/My Passport/Snowex_2023/Uncasted/Fairbanks/Creamers/*'
northslope = '/Volumes/My Passport/Snowex_2023/Uncasted/North Slope/*'

def remove_txt_files(location):
   snowpits = list_files_local(location)
   fnames = []
   for snowpit in snowpits:
      segments = list_files_local(snowpit+'/*')
      for segment in segments:
         f = list_files_local(segment+'/*/VOI*/*.txt')
         if f:
            fnames.append(f)
   return fnames
  


In [7]:
###  a function for making folders: 
def make_folder(folder_name, filenames):
    cwd = os.getcwd()
    # Create the folder if it doesn't already exist
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
        print(f"Folder '{folder_name}' created!")

    # Ensure the source file exists before attempting to copy
    for filename in filenames: 
        print(filename)
        destination_folder = folder_name   #cwd+'/'+
        if os.path.exists(destination_folder):
            shutil.copy(filename[0], destination_folder)  # Copy file
            print(f"File '{filename}' copied to '{destination_folder}'.")
        else:
            print(f"ERROR")

In [8]:
# Converts files from ANSCI to UTF8, which python can read
def UTF8_convert1(list_files, UTF8_folder):
    BLOCKSIZE = 300000 # desired size in bytes, this is 300 kB, which is larger than biggest file 

    for file in list_files: 
        name_conv = UTF8_folder + file[36:63] +'.txt'   #naming convention and moves to folder for UTF8-Files
        with codecs.open(file, "r", 'latin-1') as sourceFile:            # the 2020 data is in "mcbs"
            with codecs.open(name_conv, "w", "utf-8") as targetFile:     # convert to UTF-8
                while True:
                    contents = sourceFile.read(BLOCKSIZE)
                    if not contents:
                        break
                    targetFile.write(contents)
    

In [9]:
## CHAT_GPT Version
def UTF8_convert(list_files, UTF8_folder):
    BLOCKSIZE = 300000  # size in bytes for reading file in chunks
    
    # Ensure UTF8_folder exists
    os.makedirs(UTF8_folder, exist_ok=True)
    
    for file in list_files:
        # Generate name based on input filename
        base_name = os.path.basename(file)  # get the file name without directory path
        name_conv = os.path.join(UTF8_folder, base_name)
        
        with codecs.open(file, "r", 'latin-1') as sourceFile:
            with codecs.open(name_conv, "w", "utf-8") as targetFile:
                while True:
                    contents = sourceFile.read(BLOCKSIZE)
                    if not contents:
                        break
                    targetFile.write(contents)

In [10]:
# gives the sample depth (lower) from the file name
def sample_height(file):
    #Find scan depth,
    sc = file.split('_')
    num = float(sc[2])
    
    return num

In [11]:
#Find the term in the file (used for knowing where to start dataframe)
def find_term(term, file):
    row = 0
    file_o = open(file)
    for line in file_o:
        row += 1
        line.strip().split('/n')
        if term in line:
            return (row)
    file.close()

In [76]:
##### go through files and add to one data frame
def loop_files(files):
    # Start and end terms
    start = "2D analysis"
    end = "3D analysis"
    frames = []

    for file in files:
        # Find start and end rows
        end_row = find_term(end, file) - 4
        start_row = find_term(start, file) + 9
        nrow = end_row - start_row
        
        # Read the CSV for the specified rows
        df_int = pd.read_csv(file, skiprows=start_row, nrows=nrow)

        # Extract scan depth from filename
        match = re.search(r'_(\d+)_(\d+)_', file)
        if match:
            num1 = int(match.group(1))
            num2 = int(match.group(2))
            print(f"Scan Depth: {num1}cm to {num2}cm")
            depth = num2  # Assuming depth is derived from num2
        else:
            print("No match found")
            depth = None

        # Add depth information as a new column
        df_int['Depth (cm)'] = depth
        
        # Rename 'Unnamed: 0' column to 'File Name' if it exists
        if 'Unnamed: 0' in df_int.columns:
            df_int = df_int.rename(columns={'Unnamed: 0': 'File Name'})
        
        # Drop the first row if necessary
        df_int = df_int.drop(df_int.index[0])

        # Append to frames
        frames.append(df_int)

    # Concatenate all dataframes
    result = pd.concat(frames, ignore_index=True)

    #### finally, edit the "depth" column so it is accurate
    result['Depth (cm)'] = result['Depth (cm)'] + pd.to_numeric(result['Pos.Z'])/10
    # Move column 'C' to the first position (index 0)
    col_to_move = result.pop('Depth (cm)')  # Remove column 'C'
    result.insert(2, 'Depth (cm)', col_to_move)  # Insert 'C' at position 0

    df_sorted = result.sort_values(by='Depth (cm)')

    
    return df_sorted

In [19]:
#Main calls all other functions, takes in a file path and yes/no if you want a .csv out 
def main(path, snowpit_name, to_csv):

    list_files = list_files_local(path+'*')  
    
    #get the snowpit name of file
    snowpit = snowpit_name
    
    
    #Convert to UTF-8
    UTF8_folder = path + '/D_UTF8/'
    UTF8_convert(list_files, UTF8_folder)
    
    #Sort-Files
    UTF8_files = list_files_local(UTF8_folder+ '/*.txt') 
    UTF8_files = sorted(UTF8_files, key = sample_height) 
    #UTF8_files = sorted(list_files, key = sample_height)
    
    
    #Find Start/End of the dataframe
    start = "2D analysis"
    end = "3D analysis"
    
    #Loop through the files
    result = loop_files(UTF8_files)

    
    #If to .csv is wanted: 
    if to_csv:
        # Export our dataframe to a .csv
        result.to_csv("M_RESULTS_2D"+snowpit+".csv", index =False)
        
    return result 

In [14]:
### working on WB497 to start !
f = list_files_local('../uncasted/fairbanks/bonanza/WB497/*')

In [77]:
### step by step
path = '../uncasted/fairbanks/bonanza/WB497/'
UTF8_folder = path + 'D_UTF8/'
list_files = list_files_local(path + '*')  
    
#Convert to UTF-8 (check if exists first)
if not list_files_local(UTF8_folder):
    UTF8_convert(list_files, UTF8_folder)

#Sort-Files
UTF8_files = list_files_local(UTF8_folder+ '/*.txt') 
UTF8_files = sorted(UTF8_files, key = sample_height)

#Find Start/End of the dataframe
start = "2D analysis"
end = "3D analysis"
    

###

#Loop through the files
result = loop_files(UTF8_files)

to_csv=True
snowpit='WB497'
if to_csv:
    # Export our dataframe to a .csv
    result.to_csv("M_RESULTS_2D"+snowpit+".csv", index =False)

result


Scan Depth: 10cm to 8cm
Scan Depth: 12cm to 10cm
Scan Depth: 14cm to 12cm
Scan Depth: 16cm to 14cm
Scan Depth: 18cm to 16cm
Scan Depth: 20cm to 18cm
Scan Depth: 22cm to 20cm
Scan Depth: 24cm to 22cm


,File Name,Pos.Z,Depth (cm),Obj.N,T.Ar,Obj.Ar,Obj.Ar/T.Ar,T.Pm,Obj.Pm,Obj.Pm/Obj.Ar,...,MMI(max),MMI(min),T.Or(phi),Ecc,St.Th(pl),St.Sp(pl),St.Li.Dn(pl),FD,i.Pm,Unnamed: 39
950,wb497_10_8_rec_voi_0085.bmp,1.69003,8.169003,103.0,65.19799,30.03796,46.07190,30.23853,220.19592,7.33059,...,NaN,NaN,176.03249,0.26289,0.27283,0.31935,1.68867,1.65804,10.22781,NaN
949,wb497_10_8_rec_voi_0086.bmp,1.70991,8.170991,112.0,65.19799,30.02323,46.04932,30.23853,221.41900,7.37492,...,NaN,NaN,172.49128,0.19864,0.27119,0.31772,1.69805,1.66028,10.34611,NaN
948,wb497_10_8_rec_voi_0087.bmp,1.72979,8.172979,101.0,65.19799,30.09711,46.16263,30.23853,220.03720,7.31091,...,NaN,NaN,158.12288,0.14423,0.27356,0.31905,1.68745,1.65032,10.44552,NaN
947,wb497_10_8_rec_voi_0088.bmp,1.74968,8.174968,105.0,65.19799,30.32372,46.51021,30.23853,222.34125,7.33225,...,NaN,NaN,127.01407,0.14233,0.27277,0.31370,1.70512,1.66457,10.21957,NaN
946,wb497_10_8_rec_voi_0089.bmp,1.76956,8.176956,109.0,65.19799,30.51447,46.80277,30.23853,223.97095,7.33983,...,NaN,NaN,109.78821,0.20440,0.27249,0.30971,1.71762,1.66806,10.21134,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6075,wb497_24_22_rec_voi_0980.bmp,19.48504,23.948504,83.0,69.54238,34.75923,49.98280,31.19855,279.54790,8.04241,...,NaN,NaN,132.15232,0.35399,0.24868,0.24885,2.00991,1.72132,13.45670,NaN
6074,wb497_24_22_rec_voi_0981.bmp,19.50492,23.950492,93.0,69.54238,34.81586,50.06424,31.19855,283.72375,8.14927,...,NaN,NaN,133.40651,0.35258,0.24542,0.24479,2.03993,1.72326,13.80094,NaN
6073,wb497_24_22_rec_voi_0982.bmp,19.52480,23.952480,90.0,69.54238,34.81556,50.06381,31.19855,289.16693,8.30568,...,NaN,NaN,134.45601,0.34766,0.24080,0.24019,2.07907,1.72810,13.79512,NaN
6072,wb497_24_22_rec_voi_0983.bmp,19.54469,23.954469,85.0,69.54238,34.82965,50.08406,31.19855,293.36904,8.42297,...,NaN,NaN,134.78784,0.32943,0.23745,0.23665,2.10928,1.72975,13.95518,NaN


In [ ]:
UTF8_files = list_files_local(UTF8_folder+ '/*.txt') 
print(UTF8_files)
UTF8_files = sorted(UTF8_files, key = sample_height)
print(UTF8_files)

['../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_24_22_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_22_20_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_20_18_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_14_12_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_12_10_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_16_14_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_10_8_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_18_16_rec_voi_.batman.txt']
['../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_10_8_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_12_10_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_14_12_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_16_14_rec_voi_.batman.txt', '../uncasted/fairbanks/bonanza/WB497/D_UTF8/wb497_18_16_rec_voi_